# HealthIQ — 01: Data Cleaning

**Objective:** Inspect the raw dataset, document all quality issues, apply cleaning transformations, and save the cleaned dataset.

**Input:** `../data/raw/healthcare_patient_analytics_seaborn.csv`  
**Output:** `../data/processed/cleaned_healthcare_data.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Raw Data

In [ ]:
df_raw = pd.read_csv('../data/raw/healthcare_patient_analytics_seaborn.csv')
print(f'Shape: {df_raw.shape}')
df_raw.head(10)

## 2. Basic Inspection

In [ ]:
# Data types
df_raw.dtypes

In [ ]:
# Statistical summary
df_raw.describe(include='all')

## 3. Missing Values

In [ ]:
missing = df_raw.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing: {missing.sum()}')

**Finding:** No missing values. No imputation required.

## 4. Duplicate Rows

In [ ]:
dups = df_raw.duplicated().sum()
print(f'Duplicate rows: {dups}')

**Finding:** No duplicates found.

## 5. Categorical Column Inspection

In [ ]:
cat_cols = ['age_group', 'gender', 'region', 'department', 'treatment_type', 'visit_type']
for col in cat_cols:
    print(f'--- {col} ---')
    print(df_raw[col].value_counts())
    has_space = (df_raw[col].str.strip() != df_raw[col]).sum()
    print(f'Whitespace issues: {has_space}\n')

## 6. Numerical Validation

In [ ]:
print('Negative length_of_stay_days:', (df_raw['length_of_stay_days'] < 0).sum())
print('Negative treatment_cost:', (df_raw['treatment_cost'] < 0).sum())
print('Invalid recovery_score (<0 or >100):', ((df_raw['recovery_score'] < 0) | (df_raw['recovery_score'] > 100)).sum())
print()
print('readmission_risk range:', df_raw['readmission_risk'].min(), 'to', df_raw['readmission_risk'].max())

## 7. Outlier Detection

In [ ]:
num_cols = ['length_of_stay_days', 'treatment_cost', 'recovery_score']
for col in num_cols:
    Q1 = df_raw[col].quantile(0.25)
    Q3 = df_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    low = Q1 - 1.5 * IQR
    high = Q3 + 1.5 * IQR
    outliers = df_raw[(df_raw[col] < low) | (df_raw[col] > high)]
    print(f'{col}: {len(outliers)} outliers | bounds [{low:.2f}, {high:.2f}]')

**Decision:** All outliers are clinically plausible (e.g., LOS up to 11.9 days, costs up to \$119k). They are **retained**.

## 8. Apply Cleaning Transformations

In [ ]:
df = df_raw.copy()

# 1. Parse visit_date to datetime
df['visit_date'] = pd.to_datetime(df['visit_date'])

# 2. Strip whitespace from all categorical columns (defensive)
for col in cat_cols:
    df[col] = df[col].str.strip()

# 3. Bin readmission_risk (0–1 float) into Low / Medium / High
bins = [0, 0.3, 0.6, 1.0]
labels = ['Low', 'Medium', 'High']
df['readmission_risk'] = pd.cut(
    df['readmission_risk'], bins=bins, labels=labels, include_lowest=True
).astype(str)

print('Cleaning complete.')
print('Shape:', df.shape)
print('visit_date dtype:', df['visit_date'].dtype)
print('readmission_risk values:', df['readmission_risk'].value_counts().to_dict())

## 9. Final Dataset Summary

In [ ]:
print(f'Original rows: {len(df_raw)}')
print(f'Cleaned rows:  {len(df)}')
print(f'Rows removed:  {len(df_raw) - len(df)}')
print(f'Date range:    {df["visit_date"].min().date()} to {df["visit_date"].max().date()}')
df.head()

## 10. Save Cleaned Dataset

In [ ]:
df.to_csv('../data/processed/cleaned_healthcare_data.csv', index=False)
print('Saved: ../data/processed/cleaned_healthcare_data.csv')